In [2]:
from winnow_fcns import *
#first to siffting
#run error orrectin protocal on that
#do naive agreement

In [ ]:
def create_simulated_error(key, rng, ber=0.25, std=0.1, N=10):
    err = rng.normal(loc=0, scale=std, size=N) 
    mask = (err > ber).astype(int) 

    cooked_key = key ^ mask
    
    return cooked_key

rng = np.random.default_rng(seed=42)

init_key
key = create_simulated_key()

In [6]:
import numpy as np

# create a mock BitBuffer class for testing
class MockBitBuffer:
    def __init__(self, bits):
        self.bits = list(bits)
        self.seed = None
    def get_length(self): return len(self.bits)
    def get_bit(self, i): return self.bits[i]
    def set_bit(self, i): self.bits[i] = 1
    def clear_bit(self, i): self.bits[i] = 0
    def flip_bit(self, i): self.bits[i] = 1 - self.bits[i]
    def set_seed(self, s): self.seed = s
    def permute_buffer(self): pass # Simplified for test

#create two keys that differ by exactly one bit in the first block
alice_key = MockBitBuffer([1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0])
bob_key   = MockBitBuffer([1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0]) # index 1 is different

#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

# start first pass
alice_winnow.first_pass()
bob_winnow.first_pass()

# alice calculates syndrome for the first block 
alice_syndrome = alice_winnow.get_syndrome(0)

# bob uses Alice's syndrome to fix his key
print(f"Bob's key before fix: {bob_key.bits[:8]}")
bob_winnow.fix_with_syndrome(0, alice_syndrome)
print(f"Bob's key after fix:  {bob_key.bits[:8]}")

# verify that the keys match
if alice_key.bits == bob_key.bits:
    print(" Success! Bob corrected the error.")
else:
    print("Failure: Keys still do not match.")

Bob's key before fix: [1, 1, 1, 1, 0, 0, 1, 0]
Bob's key after fix:  [1, 0, 1, 1, 0, 0, 1, 0]
 Success! Bob corrected the error.
